# Module 07: Hyperparameter Tuning (GridSearchCV vs. RandomizedSearchCV)

Hyperparameters are the settings we configure *before* training begins (such as tree depth, learning rate, or regularization penalty).

### Two Search Paradigms:
1. **`GridSearchCV` (Brute Force)**: Evaluates every single possible combination from a predefined grid.
   - *Pros:* Guaranteed to find the best configuration inside the provided grid.
   - *Cons:* Computationally expensive ($k \times \text{combinations}$ total model fits).
2. **`RandomizedSearchCV` (Stochastic Sampling)**: Randomly samples a fixed number of combinations (`n_iter`) from specified distributions.
   - *Pros:* Drastically faster, covers a broader parameter space, scales well with many hyperparameters.
   - *Cons:* May miss the exact optimal value if `n_iter` is set too low.

### The Pipeline Syntax (`step__parameter`):
When tuning hyperparameters inside a Scikit-Learn `Pipeline`, reference parameters using double underscores:
`<step_name>__<parameter_name>` (e.g., `classifier__max_depth` or `preprocessor__num__imputer__strategy`).

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import uniform, randint

from sklearn.datasets import make_classification
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    GridSearchCV,
    RandomizedSearchCV
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

np.random.seed(42)

# 1. Synthesize binary classification data
X_raw, y_raw = make_classification(
    n_samples=1200,
    n_features=12,
    n_informative=6,
    n_redundant=2,
    n_clusters_per_class=2,
    weights=[0.75, 0.25],  # 75% class 0, 25% class 1
    random_state=42
)

feature_names = [f"feature_{i}" for i in range(12)]
X = pd.DataFrame(X_raw, columns=feature_names)
y = pd.Series(y_raw, name="churn")

# Inject occasional missing values
X.iloc[np.random.choice(len(X), size=40, replace=False), 0] = np.nan
X.iloc[np.random.choice(len(X), size=40, replace=False), 3] = np.nan

# 2. Holdout split (Test set remains completely untouched until the end)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")
print(f"Target distribution:\n{y_train.value_counts(normalize=True)}")

X_train shape: (960, 12)
X_test shape:  (240, 12)
Target distribution:
churn
0    0.748958
1    0.251042
Name: proportion, dtype: float64


In [2]:
pd.DataFrame(X_train, y_train)

,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,feature_10,feature_11
churn,,,,,,,,,,,,
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,3.633616,1.29915,1.350155,2.922885,-0.516304,-1.185888,-2.14919,0.667548,-1.487692,-0.714065,1.178419,2.560689
1,3.633616,1.29915,1.350155,2.922885,-0.516304,-1.185888,-2.14919,0.667548,-1.487692,-0.714065,1.178419,2.560689
...,...,...,...,...,...,...,...,...,...,...,...,...
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,3.633616,1.29915,1.350155,2.922885,-0.516304,-1.185888,-2.14919,0.667548,-1.487692,-0.714065,1.178419,2.560689
1,3.633616,1.29915,1.350155,2.922885,-0.516304,-1.185888,-2.14919,0.667548,-1.487692,-0.714065,1.178419,2.560689


---
## Step 1: Construct the Base Pipeline

We build a pipeline combining:
1. Imputation (`SimpleImputer`)
2. Scaling (`StandardScaler`)
3. Feature Selection (`SelectKBest`)
4. Model (`RandomForestClassifier`)

Notice the step names: `'imputer'`, `'scaler'`, `'selector'`, and `'classifier'`.

In [3]:
base_pipeline = Pipeline([
    ('imputer', SimpleImputer()),
    ('scaler', StandardScaler()),
    ('selector', SelectKBest(score_func=f_classif)),
    ('classifier', RandomForestClassifier(random_state=42))
])

# Base validation strategy
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("Base pipeline created:")
base_pipeline

Base pipeline created:


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('imputer', ...), ('scaler', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'mean'
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for missingness despite imputation. If 

---
## Step 2: Exhaustive Search with `GridSearchCV`

We define a parameter dictionary using the `<step>__<param>` syntax.
* Notice that we can tune preprocessing parameters (`imputer__strategy`, `selector__k`) alongside model parameters (`classifier__n_estimators`, `classifier__max_depth`) simultaneously.

In [4]:
param_distributions = {
    # Preprocessing distributions
    'imputer__strategy': ['mean', 'median'],
    'selector__k': randint(3, 10),                     # Discrete random uniform [3, 9]
    
    # Model distributions
    'classifier__n_estimators': randint(50, 250),      # Any integer between 50 and 250
    'classifier__max_depth': [3, 5, 8, 12, None],
    'classifier__min_samples_split': randint(2, 10),
    'classifier__max_features': ['sqrt', 'log2', 0.5]
}

random_search = RandomizedSearchCV(
    estimator=base_pipeline,
    param_distributions=param_distributions,
    n_iter=25,          # Exactly 25 trials sampled randomly
    cv=cv,
    scoring='roc_auc',
    n_jobs=-1,
    random_state=42,
    verbose=1
)

random_search.fit(X_train, y_train)

print("\n=== RANDOMIZEDSEARCHCV COMPLETED ===")
print(f"Best ROC-AUC Score: {random_search.best_score_:.4f}")
print("Best Parameters:")
for k, v in random_search.best_params_.items():
    print(f"  {k}: {v}")

Fitting 5 folds for each of 25 candidates, totalling 125 fits

=== RANDOMIZEDSEARCHCV COMPLETED ===
Best ROC-AUC Score: 0.9394
Best Parameters:
  classifier__max_depth: 12
  classifier__max_features: 0.5
  classifier__min_samples_split: 4
  classifier__n_estimators: 130
  imputer__strategy: median
  selector__k: 5


# here is a small explanation of what happened above:

Here is a breakdown of exactly what happened under the hood, why the search ran so quickly, and why those parameter names are formatted that way.

---

### 1. Why Did It Run So Fast? (`GridSearchCV` vs. `RandomizedSearchCV`)

Your intuition about testing every single combination describes **`GridSearchCV`**, not `RandomizedSearchCV`.

If the computer tested every combination in Cell 8, it would test:

* `imputer__strategy`: 2 choices
* `selector__k`: 7 choices (3 through 9)
* `classifier__n_estimators`: 200 choices (50 through 249)
* `classifier__max_depth`: 5 choices
* `classifier__min_samples_split`: 8 choices (2 through 9)
* `classifier__max_features`: 3 choices

$$\text{Total Combinations} = 2 \times 7 \times 200 \times 5 \times 8 \times 3 = 336,000 \text{ combinations!}$$


With 5-Fold Cross-Validation, that would require fitting **$1,680,000$ models**. Your machine would freeze for hours or days.

#### How `RandomizedSearchCV` Avoids This:

Notice the parameter: **`n_iter=25`**.

* It does **not** try every combination.
* It rolls a virtual die **exactly 25 times**.
* In Trial 1, it randomly pulls: `imputer='median'`, `k=7`, `n_estimators=142`, `max_depth=5`, `min_samples_split=4`, `max_features='sqrt'`. It evaluates that setup across 5 folds.
* It repeats that random draw 24 more times, tests them in parallel across your CPU cores (`n_jobs=-1`), and picks the best performer among those 25 trials. That is why it finished in seconds.

---

### 2. Why Double Underscores (`__`)?

When you pass a standalone model (like `RandomForestClassifier()`) to a tuner, the parameter name is just `n_estimators`.

However, here we passed an entire **`Pipeline`**:

```python
base_pipeline = Pipeline([
    ('imputer', SimpleImputer()),
    ('scaler', StandardScaler()),
    ('selector', SelectKBest(score_func=f_classif)),
    ('classifier', RandomForestClassifier(random_state=42))
])

```

The pipeline contains multiple objects stacked together. If you simply wrote `'max_depth'`, Scikit-Learn would not know whether `max_depth` belongs to step 1, step 2, or step 4.

Scikit-Learn uses a strict naming rule:


$$\mathbf{<step\_name>\_\_<actual\_parameter\_name>}$$

* Step name: `'imputer'` $\rightarrow$ Parameter: `'strategy'` $\rightarrow$ **`imputer__strategy`**
* Step name: `'selector'` $\rightarrow$ Parameter: `'k'` $\rightarrow$ **`selector__k`**
* Step name: `'classifier'` $\rightarrow$ Parameter: `'n_estimators'` $\rightarrow$ **`classifier__n_estimators`**

The double underscore (`__`) is the separator telling Scikit-Learn: *"Open the pipeline step named `'classifier'`, find its internal parameter named `'n_estimators'`, and change it."*

---

### 3. Explanation of All the Keys in the Dictionary

Every key maps directly to a parameter inside either `SimpleImputer`, `SelectKBest`, or `RandomForestClassifier`:

**`imputer__strategy` (`['mean', 'median']`):**
* Controls missing value replacement. Tests whether imputing with the average (`mean`) or the middle value (`median`) yields a higher cross-validation score.


**`selector__k` (`randint(3, 10)`):**
* Controls `SelectKBest`. Tests how many top features to retain (any random integer between 3 and 9) after scoring them with ANOVA $F$-values.


**`classifier__n_estimators` (`randint(50, 250)`):**
* The number of decision trees built inside the Random Forest forest. More trees generally stabilize predictions, but take longer to train.


**`classifier__max_depth` (`[3, 5, 8, 12, None]`):**
* The maximum levels deep each tree can grow.
* `3` or `5`: Shallow trees that avoid overfitting.
* `None`: Trees grow until all leaves are pure (which can overfit noise).


**`classifier__min_samples_split` (`randint(2, 10)`):**
* The minimum number of data samples required in a node before the tree is allowed to split it further. Higher values prevent the tree from creating splits based on just 1 or 2 outlier points.


**`classifier__max_features` (`['sqrt', 'log2', 0.5]`):**
* When deciding where to cut a branch, a tree does not look at every column. It samples a random subset of columns.
* `'sqrt'`: Tests $\sqrt{\text{total features}}$.
* `0.5`: Tests randomly picking 50% of the features at every split.



---

### 4. Why We Left the Base Pipeline Parameters Empty Initially

In Cell 4, we wrote:

```python
base_pipeline = Pipeline([
    ('imputer', SimpleImputer()),       # Empty parentheses
    ('scaler', StandardScaler()),
    ('selector', SelectKBest()),        # Empty parentheses
    ('classifier', RandomForestClassifier()) # Empty parentheses
])

```

We left them empty because **the search engine will overwrite them anyway**.

When you write `SimpleImputer()`, it loads Scikit-Learn's default setting (`strategy='mean'`). As soon as `random_search.fit()` runs, `RandomizedSearchCV` actively intercepts the pipeline and injects the trial values (`'median'`, `'mean'`, etc.) into those empty slots during each fold. Providing initial values inside the pipeline constructor would simply be replaced during tuning.

---
## Step 4: Comparing Exploration History (`cv_results_`)

Both search objects store detailed metrics for every trial inside `.cv_results_`, which can be converted directly into a Pandas DataFrame.

In [5]:
# Convert cv_results_ to a DataFrame
df_results = pd.DataFrame(random_search.cv_results_)

# Filter down to essential evaluation columns
display_cols = [
    'params', 
    'mean_test_score', 
    'std_test_score', 
    'rank_test_score'
]

df_top_trials = df_results[display_cols].sort_values(by='rank_test_score').head(5).reset_index(drop=True)

print("=== TOP 5 TRIALS FROM RANDOM SEARCH ===")
display(df_top_trials)

=== TOP 5 TRIALS FROM RANDOM SEARCH ===


,params,mean_test_score,std_test_score,rank_test_score
0,"{'classifier__max_depth': 12, 'classifier__max...",0.939365,0.024085,1
1,"{'classifier__max_depth': None, 'classifier__m...",0.935981,0.026042,2
2,"{'classifier__max_depth': 12, 'classifier__max...",0.935334,0.024774,3
3,"{'classifier__max_depth': None, 'classifier__m...",0.933478,0.028622,4
4,"{'classifier__max_depth': 12, 'classifier__max...",0.932446,0.032876,5


That table is the **leaderboard and lab notebook** of the search.

When you run `random_search.fit()`, it doesn’t just remember the single winner. Scikit-Learn quietly records the full history of every single trial inside a hidden dictionary called **`random_search.cv_results_`**.

In Cell 10, we simply convert that raw dictionary into a Pandas DataFrame so it is easy to read, and we sort it to view the top 5 highest-performing trials.

---

### What Each Column Means

Here is what each of those 4 displayed columns actually tells you:

1. **`rank_test_score`**:
* The leaderboard placement of that trial.
* Rank `1` is the overall champion (the combination stored in `.best_params_`). Rank `2` is the runner-up, and so on.


2. **`params`**:
* The exact "recipe" or configuration chosen for that specific trial.
* It shows the exact values picked on that spin of the wheel (e.g., `{'imputer__strategy': 'median', 'selector__k': 7, 'classifier__n_estimators': 142, ...}`).


3. **`mean_test_score`**:
* The average validation score across all 5 folds for that trial.
* Since we passed `scoring='roc_auc'`, this number is the mean ROC-AUC across the validation folds. Higher is better.


4. **`std_test_score`** (Standard Deviation):
* How stable or consistent that combination was across the 5 folds.
* A low number (e.g., `0.01`) means the model was rock-solid and scored almost the same on every fold.
* A high number (e.g., `0.08`) means the model scored great on some folds and terribly on others, which warns you that it might be erratic or unstable.



---

### Why Do We Check This in Real Life?

Looking at `cv_results_` helps you spot patterns rather than blindly accepting one winner:

* **Check for trends:** You might look at the top 5 rows and notice that *all five* used `'selector__k': 7` or `8`. That tells you with high confidence that around 7–8 features is the sweet spot for your dataset.
* **Avoid unstable winners:** If the Rank 1 trial has a score of `0.85` but an extreme `std_test_score` of `0.09`, while Rank 2 has a score of `0.849` with a `std` of `0.005`, in production you might prefer Rank 2 because it is far more reliable and less prone to random variation.

In [ ]:
# As the param names are truncated, because of the dataframe, this piece of code shows the full parameter dictionary of the rank 1 trial now, you can change that value and see the other trail values.
df_results.loc[df_results['rank_test_score'] == 1, 'params'].values[0]

{'classifier__max_depth': 12,
 'classifier__max_features': 0.5,
 'classifier__min_samples_split': 4,
 'classifier__n_estimators': 130,
 'imputer__strategy': 'median',
 'selector__k': 5}

---
## Step 5: Final Evaluation on the Locked Test Set

When tuning finishes, Scikit-Learn automatically refits the single best configuration on the **entire** training set (`X_train, y_train`). 

We call `.predict()` or `.predict_proba()` directly on the search object to test generalization against the unseen `X_test`.

In [6]:
# The search object acts directly as the refitted best model
best_model = random_search.best_estimator_

# Final audit against holdout data
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

print("=== FINAL HOLDOUT TEST EVALUATION ===")
print(f"Test ROC-AUC: {roc_auc_score(y_test, y_proba):.4f}")
print(classification_report(y_test, y_pred, target_names=['Class 0', 'Class 1']))

=== FINAL HOLDOUT TEST EVALUATION ===
Test ROC-AUC: 0.9473
              precision    recall  f1-score   support

     Class 0       0.92      0.98      0.95       180
     Class 1       0.92      0.75      0.83        60

    accuracy                           0.92       240
   macro avg       0.92      0.86      0.89       240
weighted avg       0.92      0.92      0.92       240



---
## Production Rules for Hyperparameter Optimization

1. **Use RandomizedSearch first:** Start broad with `RandomizedSearchCV` to discover which parameters actually matter, then narrow in with a small `GridSearchCV` around the best zone.
2. **Tune inside the Pipeline:** Always tune your transformers and estimators together to prevent data leakage across cross-validation folds.
3. **Control total fits:** Total fits = $\text{Number of combinations} \times \text{n\_splits}$. Keep compute times manageable by setting `n_jobs=-1`.
4. **Select the correct metric:** Match `scoring` to your target balance (`'roc_auc'` for balanced/moderate tasks, `'average_precision'` for extreme class imbalance).